In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "LargeScale_Comparison")
dataType = "SurfaceAnalysis_ERA5_Model_Comparison"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
def GetSimulationTime(RunType):
    if (RunType[0] == "TRACER") and (RunType[1] == "MOIST"):
        SimulationTime = ("2022-06-30","2022-07-03")
    return SimulationTime

RunType = ("TRACER","MOIST","NSSL")
SimulationTime = GetSimulationTime(RunType)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

RunType = ("TRACER","MOIST","TEMPO")
SimulationTime = GetSimulationTime(RunType)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, SimulationTime)

In [ ]:
####################################
#LOADING INPUT DATA

In [ ]:
#Getting InputData

#surface analysis
dataDirectory = DirectoryManager.GetDataDirectory(dataClass="Observation_Data", 
                                        ModelData=ModelData_NSSL,
                                        dataName="SurfaceAnalysis")
fileList1,filePathList1 = DirectoryManager.ListFiles(dataDirectory)

In [ ]:
dataDirectory_ERA5 = DirectoryManager.GetDataDirectory(dataClass="ERA5_Data", 
                                        ModelData=ModelData_NSSL,
                                        dataName="06-30_-_07-02_2022")
fileList2,filePathList2 = DirectoryManager.ListFiles(dataDirectory_ERA5)

In [ ]:
def Convert_ModelDate(ModelDates):
    dates = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelDates]
    return dates 
def Convert_WPCDate(WPCDates):
    dates = [datetime.strptime(t, "%H%MZ %a %b %d %Y") for t in WPCDates]
    return dates

def FindClosestDate(target_date, dates):
    """
    Compares a single datetime to a list of model datetimes.
    """
    time_deltas = [abs(target_date - dates) for dates in dates]
    
    date_index = time_deltas.index(min(time_deltas))
    return date_index

In [ ]:
####################################
#PLOTTING FUNCTIONS

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def CreateFigure(figsize=(10, 10),
                 wspace=0.1, hspace=0.1,
                 left=0.05, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a 3x3 grid of subplots with column titles in the top row.
    Allows manual adjustment of layout without tight_layout().
    """
    fig = plt.figure(figsize=figsize)

    # Apply global margins before adding subplots
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(3, 3, figure=fig)

    column_titles = ["Model Data", "Surface Analysis", "ERA5"]

    axes = []
    for i in range(3):
        row_axes = []
        for j in range(3):
            ax = fig.add_subplot(gs[i, j])
            if i == 0:
                ax.set_title(column_titles[j])
            row_axes.append(ax)
        axes.append(row_axes)

    return fig, axes


In [ ]:
def MakeContourPlot(axis,dataMatrix,multiplier,cmap='viridis'):
    axis.contourf(multiplier*dataMatrix)
    axis.colorbar(cmap=cmap)

In [ ]:
from PIL import Image

def PlotGIFFile(gif_path, ax, upscale_factor=4):
    """
    Plots the first frame of a GIF image on the given matplotlib axis,
    optionally upscaling for better display quality.

    Parameters:
    -----------
    gif_path : str
        Path to the .gif file.
    ax : matplotlib.axes.Axes
        Axis to plot the image on.
    upscale_factor : int
        Factor to upscale image dimensions (e.g., 2 = 2x larger)
    """
    # Load the first frame and convert to RGB
    img = Image.open(gif_path).convert("RGB")

    # Get original size and upscale
    if upscale_factor > 1:
        width, height = img.size
        new_size = (width * upscale_factor, height * upscale_factor)
        img = img.resize(new_size, resample=Image.BICUBIC)  # or Image.LANCZOS for slightly sharper

    # Plot image
    ax.imshow(img, aspect='auto')
    ax.axis('off')


In [ ]:
# Define map features once (efficient reuse)
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

import cartopy.mpl.geoaxes as geoaxes
def SetupPlateCarreeFeatures(ax, lat, lon, title=None):
    """
    Adds standard map features, extent, ticks, and labels to a Cartopy PlateCarree axis.

    Parameters:
    -----------
    ax : GeoAxes
        Cartopy GeoAxes to decorate.
    lat : 2D or 1D array
        Latitude values.
    lon : 2D or 1D array
        Longitude values.
    title : str, optional
        Title for the axis.
    """
    # Add map features
    ax.add_feature(COAST, linewidth=1)
    ax.add_feature(BORDERS, linewidth=0.8)
    ax.add_feature(STATES, linewidth=0.5)
    ax.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    ax.add_feature(LAKES, edgecolor="k", facecolor="none")

    # Set extent and ticks
    ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=ccrs.PlateCarree())
    ax.set_xticks(np.linspace(lon.min(), lon.max(), 5), crs=ccrs.PlateCarree())
    ax.set_yticks(np.linspace(lat.min(), lat.max(), 5), crs=ccrs.PlateCarree())
    ax.set_xlabel("Longitude (°E)")
    ax.set_ylabel("Latitude (°N)")

    if title:
        ax.set_title(title)


In [ ]:
import cartopy.io.shapereader as shpreader
from shapely.geometry import box
import numpy as np

def AddStateNames(ax, lat, lon, fontsize=8, margin_fraction=0.02):
    """
    Adds U.S. state names inside the visible lat/lon extent,
    placing text near the center of the visible part of the state.
    """
    # Define plot extent
    lat_min, lat_max = np.nanmin(lat), np.nanmax(lat)
    lon_min, lon_max = np.nanmin(lon), np.nanmax(lon)
    lat_margin = (lat_max - lat_min) * margin_fraction
    lon_margin = (lon_max - lon_min) * margin_fraction

    # Shrink bounds slightly to avoid edge overlap
    lat_min += lat_margin
    lat_max -= lat_margin
    lon_min += lon_margin
    lon_max -= lon_margin

    # Create a bounding box of visible domain
    domain_box = box(lon_min, lat_min, lon_max, lat_max)

    # Load shapefile of US states
    shpfilename = shpreader.natural_earth(resolution='50m',
                                          category='cultural',
                                          name='admin_1_states_provinces_lakes')
    reader = shpreader.Reader(shpfilename)

    for state in reader.records():
        if state.attributes['admin'] != 'United States of America':
            continue

        name = state.attributes['name']
        geom = state.geometry

        if not geom.intersects(domain_box):
            continue

        # Clip state to the visible domain
        clipped_geom = geom.intersection(domain_box)

        # Use centroid of clipped geometry for label
        label_point = clipped_geom.centroid
        x, y = label_point.x, label_point.y

        # Add label to plot
        ax.text(x, y, name,
                fontsize=fontsize,
                weight='bold',
                transform=ccrs.PlateCarree(),
                ha='center', va='center',
                color='white', zorder=10,
                bbox=dict(facecolor='black', alpha=0.3, edgecolor='none', boxstyle='round,pad=0.2'))


In [ ]:
# def MakeContourPlot(axis, dataMatrix, multiplier=1, cmap='viridis', add_colorbar=True, spacing=4):
#     """
#     Plots filled contours and black contour lines with specified spacing.
#     """
#     # Scale data
#     data_scaled = multiplier * dataMatrix

#     # Compute min/max rounded to nearest spacing
#     vmin = np.floor(data_scaled.min() / spacing) * spacing
#     vmax = np.ceil(data_scaled.max() / spacing) * spacing
#     levels = np.arange(vmin, vmax + spacing, spacing)

#     # Filled contours
#     cf = axis.contourf(data_scaled, levels=levels, cmap=cmap)

#     # Black line contours
#     cs = axis.contour(data_scaled, levels=levels, colors='black', linewidths=0.5)

#     # Optional: colorbar
#     if add_colorbar:
#         plt.colorbar(cf, ax=axis, orientation='vertical', pad=0.02)

#     # Optional: tidy up axes
#     axis.set_xticks([])
#     axis.set_yticks([])


In [ ]:
def MakeContourPlot(axis,
                    dataMatrix,
                    lat=None,
                    lon=None,
                    multiplier=1,
                    cmap='viridis',
                    add_colorbar=True,
                    spacing=4,
                    use_platecarree=False,
                    title=None):
    """
    Plots filled and line contours on either a regular or PlateCarree axis.
    Automatically upgrades axis if PlateCarree is requested but not present.
    """
    import cartopy.mpl.geoaxes as geoaxes

    # Scale data
    data_scaled = multiplier * dataMatrix

    # Compute contour levels
    vmin = np.floor(np.nanmin(data_scaled) / spacing) * spacing
    vmax = np.ceil(np.nanmax(data_scaled) / spacing) * spacing
    levels = np.arange(vmin, vmax + spacing, spacing)

    # Upgrade axis if needed
    if use_platecarree:
        if lat is None or lon is None:
            raise ValueError("lat and lon must be provided when use_platecarree=True")

        if not isinstance(axis, geoaxes.GeoAxes):
            # Replace axis with GeoAxes version
            fig = axis.figure
            pos = axis.get_position()
            fig.delaxes(axis)
            axis = fig.add_axes(pos, projection=ccrs.PlateCarree())

        # Add map features and ticks
        SetupPlateCarreeFeatures(axis, lat, lon, title=title)
        AddStateNames(axis, lat, lon)

        # Plot with geographic projection
        cf = axis.contourf(
            lon, lat, data_scaled,
            levels=levels, cmap=cmap,
            transform=ccrs.PlateCarree()
        )
        cs = axis.contour(
            lon, lat, data_scaled,
            levels=levels, colors='black', linewidths=0.5,
            transform=ccrs.PlateCarree()
        )
    else:
        # Standard plot (no projection)
        cf = axis.contourf(data_scaled, levels=levels, cmap=cmap)
        cs = axis.contour(data_scaled, levels=levels, colors='black', linewidths=0.5)
        if title:
            axis.set_title(title)

    # Optional colorbar
    if add_colorbar:
        plt.colorbar(cf, ax=axis, orientation='vertical', pad=0.02)

    # Tidy up
    # axis.set_xticks([])
    # axis.set_yticks([])

    return axis


In [ ]:
# #TESTING #*#*
# def ComputeMSLP(p_s, z_s, theta, g=9.81, R=287.0, p0_ref=100000.0, kappa=0.286):
#     """
#     Compute Mean Sea Level Pressure (MSLP) from surface pressure, height, and potential temperature.
#     """

#     def ConvertThetaToTemperature(theta, p_s):
#         """
#         Convert potential temperature to actual temperature.
#         T = θ * (p / p0)^κ
#         """
#         return theta * (p_s / p0_ref) ** kappa

#     # Compute temperature from theta
#     T = ConvertThetaToTemperature(theta, p_s)

#     # Compute MSLP
#     p0 = p_s * np.exp(g * z_s / (R * T))

#     return p0

In [ ]:
fig, axes = CreateFigure(figsize=(15,15))  # assuming this function is defined

#(1) Plotting Model Surface Pressure
model_dates = Convert_ModelDate(ModelData_NSSL.timeStrings)
WPCDates = ["1500Z SAT JUN 30 2022",
            "1500Z SAT JUL 01 2022",
            "1500Z SAT JUL 02 2022"]
for i, WPCDate in enumerate(WPCDates):
    analysis_date = Convert_WPCDate([WPCDate])[0]
    date_index = FindClosestDate(analysis_date, model_dates)
    dataMatrix = ModelData_NSSL.GetDataTimestep(t=date_index, varName = "surface_pressure")

    # ###############################
    # #CONVERTING TO MSLP #*#*TESTING
    # zgrid = ModelData_NSSL.initData['zgrid'].isel(nVertLevelsP1=0)
    # theta_data = ModelData_NSSL.GetDataTimestep(t=date_index, varName = "theta").isel(nVertLevels=0)
    # dataMatrix = ComputeMSLP(p_s=dataMatrix, z_s=zgrid, theta=theta_data)
    # ###############################
    
    multiplier=1/1e2
    MakeContourPlot(axes[i][0], dataMatrix, 
                    lat=dataMatrix['latitude'], lon=dataMatrix['longitude'],
                    multiplier=multiplier,use_platecarree=True)

#(2) Plotting Surface Analysis
gif_paths = [
    filePathList1[0],
    filePathList1[1],
    filePathList1[2]
]

for i in range(3):
    PlotGIFFile(gif_paths[i], axes[i][1])


#(3) Plotting Surface Data
filePath_geo = filePathList2[3]
data_geo = xr.open_dataset(filePath_geo)
timeStrings_ERA5 = data_geo['valid_time'].values
timeStrings_ERA5 = pd.to_datetime(timeStrings_ERA5).to_pydatetime().tolist()

for i, WPCDate in enumerate(WPCDates):
    analysis_date = Convert_WPCDate([WPCDate])[0]
    date_index = FindClosestDate(analysis_date, timeStrings_ERA5)
    dataMatrix = data_geo['z'].isel(valid_time = date_index, pressure_level=0)
    gravity = 9.81; multiplier=1/gravity
    
    MakeContourPlot(axes[i][2], dataMatrix, 
                    lat=dataMatrix['latitude'], lon=dataMatrix['longitude'],
                    multiplier=multiplier,use_platecarree=True)

In [ ]:
####################################
#TESTING

In [ ]:
# #TESTING
# #converting geopotential to pressure (FAILED)

# from scipy.interpolate import interp1d, RegularGridInterpolator
# import numpy as np

# def GetSurfacePressure_Interpolated(zs, ps, zs_model, lat_era, lon_era, lat_mpas, lon_mpas):
#     """
#     Interpolates ERA5 pressure (as a function of geopotential height) onto MPAS terrain height (zs_model),
#     using horizontal interpolation and vertical interpolation.
#     """

#     nP = zs.shape[0]
#     ny_mpas, nx_mpas = zs_model.shape

#     # Step 1: Horizontally interpolate ERA5 zs onto MPAS lat/lon grid
#     zs_interp = np.full((nP, ny_mpas, nx_mpas), np.nan)

#     for level in range(nP):
#         interp_func = RegularGridInterpolator(
#             (lat_era, lon_era),        # ✅ Fixed: lat_era and lon_era are 1D
#             zs[level, :, :],
#             bounds_error=False,
#             fill_value=np.nan
#         )
#         points = np.column_stack((lat_mpas.ravel(), lon_mpas.ravel()))
#         zs_interp[level] = interp_func(points).reshape(ny_mpas, nx_mpas)

#     # Step 2: Interpolate pressure at each MPAS gridpoint based on zs_model
#     surface_p = np.full((ny_mpas, nx_mpas), np.nan)

#     for i in range(ny_mpas):
#         for j in range(nx_mpas):
#             z_col = zs_interp[:, i, j]
#             if np.any(np.isnan(z_col)):
#                 continue
#             try:
#                 f = interp1d(z_col, ps, bounds_error=False, fill_value="extrapolate")
#                 surface_p[i, j] = f(zs_model[i, j])
#             except Exception:
#                 continue

#     return surface_p
    
# def GetInterpolatedPressure(date_index):
    
#     # ERA5 geopotential height (e.g., shape: [2, 241, 480])
#     zs = data_geo['z'].isel(valid_time=date_index, pressure_level=slice(0, 2)).values
#     ps = data_geo['pressure_level'].isel(pressure_level=slice(0, 2)).values  # [1000, 975]
    
#     # ERA5 lat/lon (1D)
#     lat_era = data_geo['latitude'].values  # shape: [241]
#     lon_era = data_geo['longitude'].values # shape: [480]
    
#     # MPAS terrain height and lat/lon
#     zs_model = ModelData_NSSL.initData['zgrid'][0].values  # shape: [500, 500]
#     lat_mpas = ModelData_NSSL.initData['latitude'].values
#     lon_mpas = ModelData_NSSL.initData['longitude'].values
    
#     # If lat_mpas and lon_mpas are 1D:
#     if lat_mpas.ndim == 1 and lon_mpas.ndim == 1:
#         lon_mpas, lat_mpas = np.meshgrid(lon_mpas, lat_mpas)
    
#     # Run interpolation
#     dataMatrix = GetSurfacePressure_Interpolated(zs, ps, zs_model, lat_era, lon_era, lat_mpas, lon_mpas)
    
#     return dataMatrix, lat_mpas, lon_mpas